# 十一、性能分析与调优：让模型跑起来之后，还要跑得快

前面十章解决了"怎么建模、怎么训练、怎么部署"的问题。这一章回答一个工程上绕不开的问题：**程序能跑，但跑得慢，慢在哪里？该优化什么？**

性能调优有一条铁律：**先测量，再优化，绝不凭感觉**。同样的模型，瓶颈可能在网络本身、数据加载、CPU/GPU 协同、甚至一行同步调用。靠猜去优化，往往事倍功半。本章的主角是 `torch.profiler`——PyTorch 官方的性能分析器，它能把每一次算子调用、CPU/GPU 耗时、内存分配都记录下来，让你"看见"时间花在了哪里。

> 本章定位：工程进阶能力。主体讲 profiler（怎么测、怎么看、怎么定位瓶颈），并给出优化工具箱的概览；`torch.compile` 因涉及独立的编译栈，放到下一章《torch.compile 深度解析》专章讲解，分布式训练作为了解内容放在本章末尾。

#### 学习目标
1. 理解"先测量再优化"的调优思路，知道 profiler 能回答什么问题；
2. 掌握 `torch.profiler.profile` 的核心参数与 `schedule` 的使用；
3. 能读懂 profiler 输出的表格，区分 CPU / GPU / 数据加载瓶颈；
4. 了解常用优化工具（`torch.compile`、混合精度、梯度累积等）各自解决什么问题，具体原理见后续章节；
5. 对分布式训练建立基本认知——个人学习者了解思路即可，不必强求实操。

#### 一、为什么需要性能分析

训练慢是最常见的抱怨，但"慢"的原因千差万别。先把可能的瓶颈分个类：

| 瓶颈类型 | 典型表现 | 该看什么指标 |
|---------|---------|------------|
| 算力瓶颈 | GPU 利用率长期接近 100% | 单个 kernel 耗时、算子是否可融合 |
| 数据瓶颈 | GPU 利用率周期性掉到很低 | DataLoader 耗时、CPU 与 GPU 是否互相等待 |
| CPU 瓶颈 | GPU 经常空闲、CPU 满载 | Python 侧开销、算子启动开销 |
| 同步瓶颈 | 出现大量 `cudaDeviceSynchronize` | `.item()`、`.cpu()`、print 等同步操作 |
| 显存瓶颈 | 频繁 OOM、显存抖动 | 内存分配/释放事件 |

`torch.profiler` 正是把这些指标量化的工具。它回答三个核心问题：

1. **时间花在哪**：哪个算子、哪行代码最耗时？
2. **CPU 和 GPU 谁在等谁**：是真算不过来，还是数据在喂不饱？
3. **优化有没有效果**：改前改后，同一算子的耗时是否真的下降？

#### 二、profiler 最小可用示例

先跑通一个最简单的例子，建立直觉。核心只有三步：**用 `profile` 包住要分析的代码 → 打印汇总表 → 导出 trace 文件**。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# 一个待分析的简单模型与一步训练
model = nn.Sequential(nn.Linear(256, 256), nn.ReLU(), nn.Linear(256, 10))
x = torch.randn(64, 256)
target = torch.randint(0, 10, (64,))
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

from torch.profiler import profile, ProfilerActivity

# 用 profile 上下文包住要分析的一段代码
with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as prof:
    for _ in range(3):
        optimizer.zero_grad()
        out = model(x)
        loss = F.cross_entropy(out, target)
        loss.backward()
        optimizer.step()

# 打印按 CUDA 自身耗时排序的汇总表
print(prof.key_averages().table(sort_by="self_cuda_time_total", row_limit=10))

上面这张表是 profiler 的核心产出。几个关键列的含义：

| 列名 | 含义 |
|------|------|
| `Name` | 算子名称 |
| `Self CPU time total` | 算子**自身**在 CPU 上的总耗时（不含它调用的下游算子） |
| `Self CUDA time total` | 算子自身在 GPU 上的总耗时 |
| `CPU time total` | 含下游调用的 CPU 总耗时 |
| `CUDA time total` | 含下游调用的 GPU 总耗时 |
| `Calls` | 调用次数 |

**看表的关键**：优先看 `Self ... time`，它才是算子自己真正吃掉的开销；`total` 列会包含子调用，容易重复计算。CPU 环境没有 CUDA，把 `sort_by` 改成 `self_cpu_time_total` 即可。

#### 三、`torch.profiler.profile` 核心参数

`profile()` 的参数不少，但常用的就这几个，按需打开——**每多开一个开关， profiling 本身的开销就大一点**：

In [ ]:
from torch.profiler import profile, ProfilerActivity, schedule, ProfilerAction

# profile 常用参数一览（这里只展示写法，不真正跑）
demo = profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],  # 采集哪些活动：CPU / CUDA / XPU
    record_shapes=True,     # 记录算子输入 shape，便于定位某个形状特别慢
    profile_memory=True,    # 跟踪张量的内存分配/释放，排查显存问题
    with_stack=True,        # 记录算子对应的源码文件与行号，直接定位到代码
    schedule=schedule(wait=1, warmup=1, active=2, repeat=1),  # 长训练时分阶段采样
)
print("profile 参数已配置（未进入上下文，不采集）")

| 参数 | 作用 | 什么时候开 |
|------|------|-----------|
| `activities` | 采集 CPU / CUDA / XPU 活动；默认 CPU + 可用的加速卡 | 总是要设 |
| `record_shapes` | 记录输入 shape | 怀疑"某个特定形状特别慢"时 |
| `profile_memory` | 跟踪内存分配/释放 | 排查显存泄漏、OOM 时 |
| `with_stack` | 记录源码行号，直接定位到代码 | 想知道耗时算子对应哪行时 |
| `schedule` | 按 iteration 分阶段采集 | 长训练任务，不想全程录 |

> 注意：`record_shapes` / `with_stack` 会让 profiler 额外持有张量引用、增加开销，所以只在排查具体问题时打开，平时用默认即可。

#### 四、长训练任务：用 schedule 控制采样

如果对一个跑几千 step 的训练任务全程 profiling，会产生海量数据、开销也大。`schedule` 可以让你按阶段采样：先跳过、再等待、再 warmup、最后录制若干 step，然后循环。

`torch.profiler.schedule(wait, warmup, active, repeat, skip_first)` 的节奏：

```
skip_first  →  wait  →  warmup  →  active  →  wait  →  warmup  →  active  → ...
  跳过         等待      预热       录制       等待      预热       录制
```

- `wait`：等待，不采集（让程序进入稳定状态）；
- `warmup`：预热，采集但不记录（消除首次编译、显存分配的抖动）；
- `active`：正式记录，每个 active 周期结束时触发 `on_trace_ready`；
- `repeat`：循环次数，0 表示一直循环。

配合 `on_trace_ready` 回调，就能在每个周期结束后自动导出 trace。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.profiler import profile, ProfilerActivity, schedule

model = nn.Linear(128, 10)
x = torch.randn(32, 128)
target = torch.randint(0, 10, (32,))
opt = torch.optim.SGD(model.parameters(), lr=0.01)

# 分阶段采样：跳过1步 → 等1步 → 预热1步 → 录2步，循环1轮
sched = schedule(wait=1, warmup=1, active=2, repeat=1)

with profile(
    activities=[ProfilerActivity.CPU],
    schedule=sched,
    on_trace_ready=lambda p: print("一个采集周期结束，可在此导出 trace"),
) as prof:
    for step in range(6):
        opt.zero_grad()
        loss = F.cross_entropy(model(x), target)
        loss.backward()
        opt.step()
        prof.step()   # 必须调用，通知 profiler 进入下一个 step

print(prof.key_averages().table(sort_by="self_cpu_time_total", row_limit=5))

> **关键点**：用了 `schedule` 必须在每个 iteration 末尾调用 `prof.step()`，否则 profiler 不知道 step 推进，schedule 不会生效。这是最容易踩的坑。

#### 五、可视化：把 trace 变成时间线

表格适合看"谁最耗时"，但看不出"CPU 和 GPU 谁在等谁"。这时把 trace 导出成 Chrome 时间线，一图看清。

```python
prof.export_chrome_trace("trace.json")
```

导出后用 Chrome 打开 `chrome://tracing`（或 `chrome://tracing` 的继任者 Perfetto UI `https://ui.perfetto.dev`），加载这个 JSON，就能看到：

- 每条水平线是一个执行流（CPU 线程、CUDA stream）；
- 每个色块是一个算子或 kernel，横向长度就是耗时；
- **看 GPU 线上有没有大段空白**：空白 = GPU 在等 CPU 喂数据或等同步，这就是优化方向。

另外，TensorBoard 也提供了 PyTorch Profiler 插件，可以把 profiler 数据直接可视化成仪表盘（需要额外安装 `torch-tb-profiler`）。

#### 六、常见瓶颈的定位思路

拿到 profiler 数据后，怎么判断瓶颈？一张对照表：

| 观察到的现象 | 大概率原因 | 优化方向 |
|------------|-----------|---------|
| GPU 利用率高，`Self CUDA time` 集中在少数 big kernel | 正常算力瓶颈 | 用 `torch.compile` 融合算子 |
| GPU 利用率周期性掉零，`aten::empty` / 数据相关算子占比高 | 数据加载跟不上 | 优化 DataLoader：`num_workers`、`pin_memory`、预取 |
| CPU 侧 `Python` / `autograd` 开销大 | Python 循环、小算子太多 | 减少小算子、批量化、上 `torch.compile` |
| 大量 `cudaDeviceSynchronize` / `aten::item` | 代码里频繁同步 | 避免训练中 `.item()`、`.cpu()`、print 张量 |
| `Self CUDA time` 远小于 `CUDA time` | kernel 启动开销占比大 | 减小 kernel 数量（融合）、用 CUDA Graph |
| 显存分配事件密集 | 频繁创建临时张量 | 复用张量、梯度检查点、减少中间变量 |

**诊断 GPU 是否喂得饱**：看 GPU 时间线空白比例。空白多说明瓶颈在 CPU/数据侧；空白少、GPU 满负荷说明瓶颈在算力本身。这一步判断对了，优化方向就不会跑偏。

#### 一个典型的同步陷阱

下面这段代码在训练循环里调用 `.item()` 把损失取成 Python 数字——这会触发 **CPU-GPU 同步**，GPU 必须等 CPU 拿到结果才能继续，profiler 里会看到大量 `cudaDeviceSynchronize`：

```python
for x, y in loader:
    loss = criterion(model(x), y)
    loss.backward()
    print(loss.item())   # 危险：每次都同步，拖慢训练
```

正确做法是**用Python的step计数器控制打印频率，或异步记录**，避免每步都读回 GPU 数值。这类问题只有 profiler 能直观暴露。

#### 七、优化工具箱概览：什么时候用什么

profiler 帮你定位了瓶颈，接下来就是从工具箱里挑合适的工具。这里只做**一句话概览**，知道每个工具解决什么问题；其中 `torch.compile` 因为涉及一整套编译栈，放到下一章《12 torch.compile 深度解析》专章展开。

| 工具 | 一句话作用 | 解决的瓶颈 |
|------|-----------|-----------|
| `torch.compile` | 一行代码把模型编译成优化 kernel，本质是算子融合 | 算子太碎、kernel 启动开销大、显存反复读写 |
| 混合精度 AMP | 用 float16/bfloat16 计算和存权重 | 显存不够、算力吃不满 |
| 梯度累积 | 多个小 batch 梯度累加后再更新 | 显存放不下大 batch |
| 梯度检查点 | 前向不存中间值、反向时重算 | 激活占显存太多 |
| 分布式训练 | 多卡分担算力/显存 | 单卡算力或显存不够 |

**和 profiler 的配合**：先用 profiler 定位瓶颈，再对症挑工具；用了工具之后再用 profiler 对比，验证是否真的变快——这就是"测量 → 优化 → 再测量"的闭环。

> `torch.compile` 的完整原理（TorchDynamo 图捕获、AOTAutograd、Inductor、图断裂、guard、三种 mode、调试方法）见下一章《12 torch.compile 深度解析》。

#### 八、分布式训练概览（了解即可）

当单卡算力或显存不够时，就需要分布式训练。对多数个人学习者来说，**没有多卡多机条件，这一部分了解思路即可，不必强求实操**——知道它是什么、解决什么问题、大致怎么用，将来有机会上手时能看懂就行。

#### 8.1 两种最基本的并行思路

| 方式 | 核心思想 | 适用场景 |
|------|---------|---------|
| **数据并行（DDP）** | 每个 GPU 都有完整模型，把数据切分给各卡，各算各的梯度，再**把所有卡的梯度平均同步** | 最常见，单卡能放下模型时用 |
| **模型并行 / 张量并行** | 一个模型太大，把模型本身拆到多张卡上 | 单卡放不下大模型（如 LLM） |

目前最常用的是 **DDP（DistributedDataParallel）**——数据并列的工业标准。

#### 8.2 DDP 的训练闭环

```
① 每个进程（通常是每张卡）加载一份完整模型
② 同一批数据被切分，每个进程处理自己那份
③ 各自前向 + 反向，算出本地梯度
④ all_reduce：把所有进程的梯度求平均（这一步是分布式的核心）
⑤ 每个进程用相同的平均梯度更新参数 → 各卡参数始终一致
```

关键点在第 ④ 步的**梯度同步**——它保证多卡训练在数学上等价于单卡用大 batch 训练。

#### 8.3 最小代码形态（了解结构即可）

用 `torchrun` 启动时，每个进程拿到自己的 `rank` 和世界大小：

```python
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

dist.init_process_group("nccl")          # 初始化进程组，NCCL 是 GPU 通信后端
local_rank = int(os.environ["LOCAL_RANK"])
torch.cuda.set_device(local_rank)
model = DDP(MyModel().to(local_rank), device_ids=[local_rank])  # 包装模型
# ... 正常训练，DDP 会在 backward 时自动做梯度同步 ...
```

启动命令：`torchrun --nproc_per_node=2 train.py`（单机两卡）。

#### 8.4 个人学习者的现实定位

- **单卡够用就别上分布式**：DDP 会引入进程通信、数据切分、同步等复杂度，单卡能跑时没必要；
- **显存不够的替代方案**：先试梯度累积、混合精度（`torch.cuda.amp.autocast`）、梯度检查点，这些单卡就能做；
- **真要学 DDP**：用云平台的按小时GPU，或先用单机多进程（哪怕只有一张卡，起两个进程也能跑通 DDP 流程）练手；
- **更上层的选择**：PyTorch Lightning、HuggingFace Trainer 都把 DDP 封装好了，改个参数就能多卡训练——这层抽象对个人很友好。

#### 九、本章总结

| 你学到的 | 一句话要点 |
|---------|-----------|
| 调优铁律 | **先测量，再优化**——绝不凭感觉 |
| profiler 作用 | 量化算子耗时、CPU/GPU 协同、内存分配，回答"时间花在哪" |
| 核心接口 | `profile(activities=...)` + `key_averages().table()` + `export_chrome_trace()` |
| 长训练采样 | `schedule(wait, warmup, active)` 分阶段录制，务必配 `prof.step()` |
| 读表关键 | 看 `Self ... time` 而非 `total`，避免重复计算 |
| 看时间线 | GPU 线上空白多 = 数据/CPU 喂不饱；满负荷 = 算力瓶颈 |
| 优化工具箱 | compile 治碎算子、AMP 治显存、梯度累积治大 batch、检查点治激活；`torch.compile` 详见第 12 章 |
| 调优闭环 | 测量 → 定位瓶颈 → 针对性优化 → 再测量验证 |
| 分布式 | 数据并行(DDP)最常用，核心是梯度 all_reduce；个人了解思路即可 |

**最终心法**：性能调优不是堆技巧，而是一套科学方法——用 profiler 把"慢"这个模糊感受，变成"某个算子在某种情况下耗时多少"的精确事实，再对症下药。`torch.compile`、混合精度、梯度累积、分布式，都是工具箱里的具体工具；什么时候用哪个，由测量结果说了算。